<a href="https://colab.research.google.com/github/chk-33/large-language-modelloing/blob/main/flash%20attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(
    url,
    "tinyshakespeare.txt"
)
with open(
    "tinyshakespeare.txt",
    "r",
    encoding="utf-8"
) as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {
    ch: i
    for i, ch in enumerate(chars)
}
itos = {
    i: ch
    for i, ch in enumerate(chars)
}
data = torch.tensor(
    [stoi[ch] for ch in text],
    dtype=torch.long
)
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

# Model configuration
vocab_size = 65
d_model = 128
num_layers = 2
num_heads = 2
kv_heads = 2
d_ff = 512
num_experts = 4
top_k = 2
max_seq_len = 256
dropout = 0.1

# Training configuration
batch_size = 32
block_size = 64
max_steps = 2500
learning_rate = 1e-3
min_learning_rate = 1e-4
warmup_steps = 100
weight_decay = 0.1
grad_clip = 1.0
eval_interval = 250
eval_iters = 20

# MoE configuration
capacity_factor = 1.25

# Auxiliary loss weight
aux_loss_weight = 0.01

# ============================================================
# SHAKESPEARE BATCH LOADER
# ============================================================

def get_batch(
    data,
    batch_size,
    block_size,
    device
):
    ix = torch.randint(
        0,
        len(data) - block_size - 1,
        (batch_size,)
    )
    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])
    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])
    return x.to(device), y.to(device)

class RMSNorm(nn.Module):
    def __init__(
        self,
        d_model,
        eps=1e-6
    ):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(
            torch.ones(d_model)
        )

    def forward(self, x):
        rms = torch.sqrt(
            torch.mean(
                x ** 2,
                dim=-1,
                keepdim=True
            )
            + self.eps
        )
        x = x / rms
        return self.weight * x

class RoPE(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len=2048,
        base=10000
    ):
        super().__init__()
        assert head_dim % 2 == 0
        self.head_dim = head_dim
        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )
        positions = torch.arange(
            max_seq_len
        ).float()
        freqs = torch.outer(
            positions,
            inv_freq
        )
        self.register_buffer(
            "cos",
            freqs.cos()
        )
        self.register_buffer(
            "sin",
            freqs.sin()
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        S = q.size(-2)
        positions = torch.arange(
            position_offset,
            position_offset + S,
            device=q.device
        )
        cos = self.cos[positions]
        sin = self.sin[positions]
        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)
        q1 = q[..., ::2]
        q2 = q[..., 1::2]
        k1 = k[..., ::2]
        k2 = k[..., 1::2]
        q_rotated = torch.stack(
            [
                q1 * cos - q2 * sin,
                q1 * sin + q2 * cos
            ],
            dim=-1
        )
        k_rotated = torch.stack(
            [
                k1 * cos - k2 * sin,
                k1 * sin + k2 * cos
            ],
            dim=-1
        )
        q_rotated = q_rotated.flatten(
            -2
        )
        k_rotated = k_rotated.flatten(
            -2
        )
        return (
            q_rotated,
            k_rotated
        )

class GQA(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        kv_heads,
        max_seq_len=2048,
        dropout=0.0
    ):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % kv_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.kv_heads = kv_heads
        self.head_dim = (
            d_model // num_heads
        )
        self.groups = (
            num_heads // kv_heads
        )
        self.q_proj = nn.Linear(
            d_model,
            num_heads * self.head_dim,
            bias=False
        )
        self.k_proj = nn.Linear(
            d_model,
            kv_heads * self.head_dim,
            bias=False
        )
        self.v_proj = nn.Linear(
            d_model,
            kv_heads * self.head_dim,
            bias=False
        )
        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )
        self.rope = RoPE(
            self.head_dim,
            max_seq_len
        )
        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x, kv_cache=None, position_offset=0):
        B, S, D = x.shape
        q = self.q_proj(x)
        q = q.view(
            B,
            S,
            self.num_heads,
            self.head_dim
        )
        q = q.transpose(
            1,
            2
        )
        k = self.k_proj(x)
        k = k.view(
            B,
            S,
            self.kv_heads,
            self.head_dim
        )
        k = k.transpose(
            1,
            2
        )
        v = self.v_proj(x)
        v = v.view(
            B,
            S,
            self.kv_heads,
            self.head_dim
        )
        v = v.transpose(
            1,
            2
        )
        q, k = self.rope(
            q,
            k,
            position_offset=position_offset
        )

        if kv_cache is not None:
            k_prev, v_cache_prev = kv_cache
            k = torch.cat([k_prev, k], dim=-2)
            v = torch.cat([v_cache_prev, v], dim=-2)

        new_kv_cache = (k, v)

        k_rep = k.repeat_interleave(
            self.groups,
            dim=1
        )
        v_rep = v.repeat_interleave(
            self.groups,
            dim=1
        )
        scores = (
            q @ k_rep.transpose(
                -2,
                -1
            )
        )
        scores = (
            scores
            / math.sqrt(
                self.head_dim
            )
        )
        S_kv = k.size(-2)
        query_positions = (
             position_offset
               + torch.arange(
                S,
             device=x.device
                 )
                  )

        key_positions = torch.arange(
        S_kv,
        device=x.device
        )

        mask = (
        key_positions.unsqueeze(0)
        > query_positions.unsqueeze(1)
    )

        if kv_cache is None:
            out = F.scaled_dot_product_attention(
                q,
                k_rep,
                v_rep,
                dropout_p=(
                    self.dropout.p
                    if self.training
                    else 0.0
                ),
                is_causal=True
            )
        else:
            S_kv = k.size(-2)
            query_positions = (
                position_offset
                + torch.arange(
                    S,
                    device=x.device
                )
            )
            key_positions = torch.arange(
                S_kv,
                device=x.device
            )
            attention_mask = (
                key_positions.unsqueeze(0)
                <= query_positions.unsqueeze(1)
            )
            out = F.scaled_dot_product_attention(
                q,
                k_rep,
                v_rep,
                attn_mask=attention_mask,
                dropout_p=(
                    self.dropout.p
                    if self.training
                    else 0.0
                ),
                is_causal=False
            )
        out = out.transpose(
            1,
            2
        ).contiguous()
        out = out.view(
            B,
            S,
            D
        )
        out = self.out_proj(
            out
        )
        return out, new_kv_cache

class Expert(nn.Module):
    def __init__(
        self,
        d_model,
        d_ff
    ):
        super().__init__()
        self.gate_proj = nn.Linear(
             d_model,
            d_ff,
            bias=False
        )
        self.value_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False
        )
        self.output_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False
        )

    def forward(self, x):
        gate_pre = self.gate_proj(x)
        value = self.value_proj(x)
        gate = F.silu(gate_pre)
        hidden = gate * value
        output = self.output_proj(hidden)
        return output

# ============================================================
# LOAD BALANCING LOSS
# ============================================================

def load_balancing_loss(
    router_probs,
    top_indices,
    num_experts
):
    P = router_probs.mean(
        dim=(0, 1)
    )
    assignment_mask = F.one_hot(
        top_indices,
        num_classes=num_experts
    ).float()
    f = assignment_mask.mean(
        dim=(0, 1, 2)
    )
    loss = (
        num_experts
        * torch.sum(f * P)
    )
    return loss

# ============================================================
# MoE DISPATCHER
# ============================================================

class MoEDispatcher:
    def __init__(
        self,
        num_experts,
        top_k,
        capacity
    ):
        self.num_experts = num_experts
        self.top_k = top_k
        self.capacity = capacity

    def dispatch(
        self,
        x,
        top_indices
    ):
        B, S, D = x.shape
        T = B * S
        x_flat = x.reshape(T, D)
        expert_inputs = []
        token_indices_per_expert = []
        k_indices_per_expert = []
        dropped_assignments = 0

        for expert_id in range(
            self.num_experts
        ):
            mask = (top_indices == expert_id)
            mask = mask.reshape(T, self.top_k)
            token_indices, k_indices = torch.where(mask)
            num_assignments = token_indices.numel()

            if num_assignments > self.capacity:
                dropped_assignments += (
                    num_assignments
                    - self.capacity
                )
                token_indices = token_indices[:self.capacity]
                k_indices = k_indices[:self.capacity]

            expert_input = x_flat[token_indices]
            expert_inputs.append(expert_input)
            token_indices_per_expert.append(token_indices)
            k_indices_per_expert.append(k_indices)

        return (
            expert_inputs,
            token_indices_per_expert,
            k_indices_per_expert,
            dropped_assignments
        )

# ============================================================
# Mixture of Experts
# ============================================================

class MoE(nn.Module):
    def __init__(
        self,
        d_model,
        d_ff,
        num_experts=4,
        top_k=2,
        capacity_factor=1.25
    ):
        super().__init__()
        assert num_experts >= 1
        assert 1 <= top_k <= num_experts
        assert capacity_factor > 0
        self.d_model = d_model
        self.d_ff = d_ff
        self.num_experts = num_experts
        self.top_k = top_k
        self.capacity_factor = capacity_factor

        self.router = nn.Linear(
            d_model,
            num_experts,
            bias=False
        )
        self.experts = nn.ModuleList(
            [
                Expert(d_model, d_ff)
                for _ in range(num_experts)
            ]
        )

    def forward(self, x):
        B, S, D = x.shape
        assert D == self.d_model
        T = B * S

        router_logits = self.router(x)
        router_probs = F.softmax(router_logits, dim=-1)
        top_weights, top_indices = torch.topk(
            router_probs,
            k=self.top_k,
            dim=-1
        )
        top_weights = (
            top_weights
            / top_weights.sum(
                dim=-1,
                keepdim=True
            ).clamp_min(1e-9)
        )
        capacity = math.ceil(
            (
                T * self.top_k
                / self.num_experts
            )
            * self.capacity_factor
        )
        dispatcher = MoEDispatcher(
            num_experts=self.num_experts,
            top_k=self.top_k,
            capacity=capacity
        )
        (
            expert_inputs,
            token_indices_per_expert,
            k_indices_per_expert,
            dropped_assignments
        ) = dispatcher.dispatch(x, top_indices)

        expert_outputs = []
        for expert_id in range(self.num_experts):
            expert_input = expert_inputs[expert_id]
            if expert_input.numel() == 0:
                expert_output = expert_input
            else:
                expert_output = self.experts[expert_id](expert_input)
            expert_outputs.append(expert_output)

        x_flat = x.reshape(T, D)
        output = torch.zeros_like(x_flat)
        top_weights_flat = top_weights.reshape(T, self.top_k)

        for expert_id in range(self.num_experts):
            token_indices = token_indices_per_expert[expert_id]
            k_indices = k_indices_per_expert[expert_id]
            if token_indices.numel() == 0:
                continue
            expert_output = expert_outputs[expert_id]
            weights = top_weights_flat[token_indices, k_indices]
            weights = weights.unsqueeze(-1)
            weighted_output = expert_output * weights
            output.index_add_(0, token_indices, weighted_output)

        output = output.view(B, S, D)
        return (
            output,
            router_logits,
            router_probs,
            top_indices,
            dropped_assignments
        )

class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model=128,
        num_heads=4,
        kv_heads=2,
        d_ff=512,
        num_experts=4,
        top_k=2,
        max_seq_len=2048,
        dropout=0.0
    ):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = GQA(
            d_model=d_model,
            num_heads=num_heads,
            kv_heads=kv_heads,
            max_seq_len=max_seq_len,
            dropout=dropout
        )
        self.norm2 = RMSNorm(d_model)
        self.moe = MoE(
            d_model=d_model,
            d_ff=d_ff,
            num_experts=num_experts,
            top_k=top_k,
            capacity_factor=capacity_factor
        )

    def forward(self, x, kv_cache=None, position_offset=0):
        residual = x
        x = self.norm1(x)
        x, new_kv_cache = self.attention(x, kv_cache=kv_cache, position_offset=position_offset)
        x = residual + x

        residual = x
        x = self.norm2(x)
        (
            moe_output,
            router_logits,
            router_probs,
            top_indices,
            dropped_tokens
        ) = self.moe(x)
        x = residual + moe_output

        aux_loss = load_balancing_loss(
            router_probs,
            top_indices,
            self.moe.num_experts
        )
        return (
            x,
            new_kv_cache,
            aux_loss,
            router_logits,
            router_probs,
            top_indices,
            dropped_tokens
        )

# ============================================================
# COMPLETE TRANSFORMER MODEL
# ============================================================

class TransformerModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        kv_heads,
        d_ff,
        num_experts,
        top_k,
        max_seq_len,
        dropout=0.1
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    kv_heads=kv_heads,
                    d_ff=d_ff,
                    num_experts=num_experts,
                    top_k=top_k,
                    max_seq_len=max_seq_len,
                    dropout=dropout
                )
                for _ in range(num_layers)
            ]
        )
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids, targets=None, kv_caches=None, position_offset=0):
        x = self.embedding(input_ids)
        total_aux_loss = torch.zeros((), device=x.device)
        total_dropped_tokens = 0
        router_information = []
        new_kv_caches = []

        for i, block in enumerate(self.blocks):
            kv_cache = kv_caches[i] if kv_caches is not None else None
            (
                x,
                new_kv_cache,
                aux_loss,
                router_logits,
                router_probs,
                top_indices,
                dropped_tokens
            ) = block(x, kv_cache=kv_cache, position_offset=position_offset)
            new_kv_caches.append(new_kv_cache)
            total_aux_loss = total_aux_loss + aux_loss
            total_dropped_tokens += dropped_tokens
            router_information.append({
                "logits": router_logits,
                "probs": router_probs,
                "top_indices": top_indices
            })

        x = self.final_norm(x)
        logits = self.lm_head(x)
        lm_loss = None

        if targets is not None:
            lm_loss = F.cross_entropy(
                logits.reshape(-1, self.vocab_size),
                targets.reshape(-1)
            )

        loss = None
        if lm_loss is not None:
            loss = (
                lm_loss
                + aux_loss_weight
                * total_aux_loss
            )

        return (
            logits,
            loss,
            lm_loss,
            total_aux_loss,
            total_dropped_tokens,
            router_information,
            new_kv_caches
        )

# ============================================================
# CREATE MODEL & CHECKPOINT RECOVERY
# ============================================================

model = TransformerModel(
    vocab_size=vocab_size,
    d_model=d_model,
    num_layers=num_layers,
    num_heads=num_heads,
    kv_heads=kv_heads,
    d_ff=d_ff,
    num_experts=num_experts,
    top_k=top_k,
    max_seq_len=max_seq_len,
    dropout=dropout
).to(device)

try:
    checkpoint = torch.load("best_model.pt", map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print("Successfully loaded best model weights from checkpoint.")
except Exception as e:
    print("Could not find or load checkpoint; proceeding with initialized model. Details:", e)

# ============================================================
@torch.no_grad()
def evaluate_model(
    model,
    data,
    batch_size,
    block_size,
    eval_iters,
    device
):
    model.eval()

    losses = []

    for _ in range(eval_iters):

        x, y = get_batch(
            data,
            batch_size,
            block_size,
            device
        )

        (
            logits,
            loss,
            lm_loss,
            aux_loss,
            dropped_tokens,
            router_information,
            kv_caches
        ) = model(
            x,
            targets=y
        )

        losses.append(
            lm_loss.item()
        )

    model.train()

    mean_loss = sum(losses) / len(losses)

    perplexity = math.exp(mean_loss)

    return mean_loss, perplexity
# LEARNING RATE SCHEDULE
# ============================================================

def get_lr(step, max_steps, lr_max, lr_min, warmup_steps):
    if step < warmup_steps:
        return lr_max * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return lr_min + cosine * (lr_max - lr_min)

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(model, data, batch_size, block_size, device, eval_iters):
    model.eval()
    total_losses = []
    for _ in range(eval_iters):
        input_ids, targets = get_batch(data, batch_size, block_size, device)
        _, loss, _, _, _, _, _ = model(input_ids, targets)
        total_losses.append(loss.item())
    model.train()
    return sum(total_losses) / len(total_losses)

# ============================================================
# TRAINING PIPELINE
# ============================================================

print("Training the model to minimize errors...")
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
for step in range(max_steps):
    lr = get_lr(step, max_steps, learning_rate, min_learning_rate, warmup_steps)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    input_ids, targets = get_batch(train_data, batch_size, block_size, device)
    logits, loss, lm_loss, _, _, _, _ = model(input_ids, targets)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_interval == 0 or step == max_steps - 1:
        val_loss = evaluate(model, val_data, batch_size, block_size, device, eval_iters)
        print(f"Step {step:4d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f}")

# Save trained state
torch.save({"model_state_dict": model.state_dict()}, "best_model.pt")

# ============================================================
# RE-ENGINEERED GENERATION WITH KV CACHING & EXPONENTIAL REPETITION PENALTY
# ============================================================

@torch.no_grad()
def generate(
    model,
    input_ids,
    max_new_tokens,
    max_seq_len,
    temperature=1.0,
    top_p=0.92,
    top_k=10,
    repetition_penalty=1.5
):
    model.eval()
    kv_caches = None

    # Process the initial prompt to populate the KV cache
    logits, _, _, _, _, _, kv_caches = model(input_ids, kv_caches=kv_caches, position_offset=0)
    logits = logits[:, -1, :]

    # Apply repetition penalty to initial logits if applicable
    if repetition_penalty != 1.0:
        for i in range(input_ids.size(0)):
            unique_tokens, counts = torch.unique(input_ids[i], return_counts=True)
            for token, count in zip(unique_tokens, counts):
                token_id = token.item()
                logits[i, token_id] -= float(count) * repetition_penalty

    logits = logits / temperature
    if top_k is not None:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')

    if top_p is not None:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        for i in range(logits.size(0)):
            indices_to_remove = sorted_indices[i][sorted_indices_to_remove[i]]
            logits[i, indices_to_remove] = float("-inf")

    next_token = torch.argmax(logits, dim=-1, keepdim=True)
    input_ids = torch.cat((input_ids, next_token), dim=1)

    for step in range(1, max_new_tokens):
        # For generation, we only pass the *last* token
        curr_input = input_ids[:, [-1]]
        position_offset = input_ids.size(1) - 2

        logits, _, _, _, _, _, kv_caches = model(
            curr_input,
            kv_caches=kv_caches,
            position_offset=position_offset
        )
        logits = logits[:, -1, :]

        if repetition_penalty != 1.0:
            for i in range(input_ids.size(0)):
                unique_tokens, counts = torch.unique(input_ids[i], return_counts=True)
                for token, count in zip(unique_tokens, counts):
                    token_id = token.item()
                    logits[i, token_id] -= float(count) * repetition_penalty

        logits = logits / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            for i in range(logits.size(0)):
                indices_to_remove = sorted_indices[i][sorted_indices_to_remove[i]]
                logits[i, indices_to_remove] = float("-inf")

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.argmax(
            logits,
            dim=-1,
            keepdim=True,
        )
        input_ids = torch.cat((input_ids, next_token), dim=1)
    return input_ids

# ============================================================
# EXAMPLE INFERENCE USAGE
# ============================================================
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
prompt_text = "First Citizen:"
prompt_tokens = torch.tensor([encode(prompt_text)], dtype=torch.long, device=device)

MAX_NEW_TOKENS = 120
TEMPERATURE = 0.85
TOP_K = 35
TOP_P=0.90

generated_indices = generate(
    model=model,
    input_ids=prompt_tokens,
    max_new_tokens=MAX_NEW_TOKENS,
    max_seq_len=max_seq_len,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=1.5
)
generated_text = decode(generated_indices[0].tolist())
print("================ GENERATED OUTPUT =================")
print(generated_text)
val_loss, val_perplexity = evaluate_model(
    model,
    val_data,
    batch_size,
    block_size,
    eval_iters,
    device
)

print("Validation Loss:", val_loss)
print("Validation Perplexity:", val_perplexity)
@torch.no_grad()
def evaluate_expert_usage(
    model,
    data,
    batch_size,
    block_size,
    eval_iters,
    device
):
    model.eval()

    num_experts = model.blocks[0].moe.num_experts

    expert_counts = torch.zeros(
        num_experts,
        device=device
    )

    total_assignments = 0

    total_dropped = 0

    for _ in range(eval_iters):

        x, y = get_batch(
            data,
            batch_size,
            block_size,
            device
        )

        (
            logits,
            loss,
            lm_loss,
            aux_loss,
            dropped_tokens,
            router_information,
            kv_caches
        ) = model(x)

        for info in router_information:

            top_indices = info["top_indices"]

            for expert_id in range(num_experts):

                expert_counts[expert_id] += (
                    (top_indices == expert_id)
                    .sum()
                )

            total_assignments += (
                top_indices.numel()
            )

        total_dropped += dropped_tokens

    usage_percent = (
        expert_counts
        / total_assignments
        * 100
    )

    drop_rate = (
        total_dropped
        / total_assignments
        * 100
    )

    model.train()

    return usage_percent, drop_rate
expert_usage, drop_rate = evaluate_expert_usage(
    model,
    val_data,
    batch_size,
    block_size,
    eval_iters,
    device
)

print("\n===== MOE EVALUATION =====")

for i, usage in enumerate(expert_usage):
    print(
        f"Expert {i}: {usage.item():.2f}%"
    )

print(
    f"Dropped Assignment Rate: "
    f"{drop_rate:.2f}%"
)

Could not find or load checkpoint; proceeding with initialized model. Details: [Errno 2] No such file or directory: 'best_model.pt'
Training the model to minimize errors...
Step    0 | Train Loss: 122.8010 | Val Loss: 122.3959
Step  250 | Train Loss: 2.4348 | Val Loss: 2.3744
Step  500 | Train Loss: 2.1682 | Val Loss: 2.1349
Step  750 | Train Loss: 2.0053 | Val Loss: 2.0416
Step 1000 | Train Loss: 1.7999 | Val Loss: 1.9836
Step 1250 | Train Loss: 1.7830 | Val Loss: 1.9566
Step 1500 | Train Loss: 1.8328 | Val Loss: 1.9157
Step 1750 | Train Loss: 1.6679 | Val Loss: 1.8778
Step 2000 | Train Loss: 1.7104 | Val Loss: 1.8572
Step 2250 | Train Loss: 1.6263 | Val Loss: 1.8663
Step 2499 | Train Loss: 1.6461 | Val Loss: 1.8230
================ GENERATED OUTPUT =================
First Citizen:
I'll way, buthough.

PAULINA:
To make,--ford; whence!'s; and,-ppoy?
What!'s.
BROMEO:
Heck, myourngmblives?'dvifff.
Your
Validation Loss: 1.804826992750168
Validation Perplexity: 6.078919660367084

===== MOE

In [2]:
import time
import math
import torch
import torch.nn.functional as F

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

B = 1
H = 2
S = 128
D = 64

q = torch.randn(
    B, H, S, D,
    device=device
)

k = torch.randn(
    B, H, S, D,
    device=device
)

v = torch.randn(
    B, H, S, D,
    device=device
)

def manual_attention(q, k, v):
    scores = (
        q @ k.transpose(-2, -1)
    )

    scores = (
        scores
        / math.sqrt(q.size(-1))
    )

    S_len = q.size(-2)

    mask = torch.triu(
        torch.ones(
            S_len,
            S_len,
            device=q.device
        ),
        diagonal=1
    ).bool()

    scores = scores.masked_fill(
        mask,
        float("-inf")
    )

    weights = F.softmax(
        scores,
        dim=-1
    )

    return weights @ v

# Benchmark SDPA
start = time.perf_counter()
for _ in range(100):
    output_sdpa = F.scaled_dot_product_attention(
        q,
        k,
        v,
        is_causal=True
    )
sdpa_time = time.perf_counter() - start

print(
    "SDPA time:",
    f"{sdpa_time:.6f}",
    "seconds"
)

# Benchmark Manual Attention
start = time.perf_counter()
for _ in range(100):
    output_manual = manual_attention(q, k, v)
manual_time = time.perf_counter() - start

print(
    "Manual time:",
    f"{manual_time:.6f}",
    "seconds"
)

# Verify correctness
assert torch.allclose(output_sdpa, output_manual, atol=1e-5), "Outputs do not match!"

speedup = manual_time / sdpa_time
improvement = ((manual_time - sdpa_time) / manual_time) * 100

print(
    f"Speedup: {speedup:.2f}x"
)
print(
    f"Improvement: {improvement:.2f}%"
)
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        ),
        "GB"
    )
else:
    print("GPU not detected. Check your Colab runtime settings.")

SDPA time: 0.002333 seconds
Manual time: 0.033434 seconds
Speedup: 14.33x
Improvement: 93.02%
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [10]:
import os
from google.colab import drive

drive.mount("/content/drive")

print("Current directory:", os.getcwd())
print("Current files:", os.listdir("/content")[:30])
print("Google Drive folders:", os.listdir("/content/drive/MyDrive")[:30])
import torch
import os

checkpoint_path = "/content/best_model.pt"

print("Checkpoint exists:", os.path.exists(checkpoint_path))
print("Checkpoint size (MB):", round(
    os.path.getsize(checkpoint_path) / (1024 * 1024), 2
))

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys())[:30])
state_dict = checkpoint["model_state_dict"]

print("Number of saved tensors:", len(state_dict))

for name, tensor in list(state_dict.items())[:20]:
    print(f"{name:60s} {tuple(tensor.shape)}")
state_dict = checkpoint["model_state_dict"]

for name, tensor in state_dict.items():
    print(f"{name:65s} {tuple(tensor.shape)}")

print("\nTotal saved tensors:", len(state_dict))
print("LoRA tensors:", [
    name for name in state_dict
    if ".A" in name or ".B" in name
    or "lora" in name.lower()
])
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    "/content/best_model.pt",
    map_location="cpu",
    weights_only=True
)

state_dict = checkpoint["model_state_dict"]

# Verify compatibility before loading anything
current_state = model.state_dict()

missing_keys = set(current_state) - set(state_dict)
unexpected_keys = set(state_dict) - set(current_state)

shape_mismatches = [
    (name, tuple(current_state[name].shape), tuple(state_dict[name].shape))
    for name in current_state.keys() & state_dict.keys()
    if current_state[name].shape != state_dict[name].shape
]

print("Missing keys:", sorted(missing_keys))
print("Unexpected keys:", sorted(unexpected_keys))
print("Shape mismatches:", shape_mismatches)

if not missing_keys and not unexpected_keys and not shape_mismatches:
    model.load_state_dict(state_dict, strict=True)
    model = model.to(device)
    print("Checkpoint loaded successfully.")
    print("Training device:", device)
else:
    print("Checkpoint NOT loaded. Resolve the differences first.")
import torch

@torch.no_grad()
def evaluate(model, split="val", eval_iters=100):
    model.eval()
    losses = []
    # Resolve the data variable to pass to get_batch
    eval_data = val_data if split == "val" else train_data

    for _ in range(eval_iters):
        x, y = get_batch(eval_data, batch_size, block_size, device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits, loss, lm_loss, total_aux_loss, total_dropped_tokens, router_information, new_kv_caches = model(x, targets=y)

        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)

val_loss = evaluate(model, "val")
perplexity = torch.exp(torch.tensor(val_loss)).item()

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation perplexity: {perplexity:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current directory: /content
Current files: ['.config', 'tinyshakespeare.txt', 'best_model.pt', 'drive', 'sample_data']
Google Drive folders: ['assignment.pdf', 'Untitled form.gform', 'IMG_0281.JPG.gdoc', 'Project proposal.gdoc', 'Resume.gdoc', 'Classroom', 'Distrubuted computing assignment.gdoc', 'Untitled document.gdoc', 'CHARLES KILONZI CV.pdf', 'Colab Notebooks']
Checkpoint exists: True
Checkpoint size (MB): 6.68
Checkpoint type: <class 'dict'>
Checkpoint keys: ['model_state_dict']
Number of saved tensors: 45
embedding.weight                                             (65, 128)
blocks.0.norm1.weight                                        (128,)
blocks.0.attention.q_proj.weight                             (128, 128)
blocks.0.attention.k_proj.weight                             (128, 128)
blocks.0.attention.v_proj.weight                             (128, 128

In [13]:
import os
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Preserve the verified starting checkpoint in Google Drive.
SAVE_DIR = "/content/drive/MyDrive/TransformerProject"
os.makedirs(SAVE_DIR, exist_ok=True)

BACKUP_PATH = os.path.join(SAVE_DIR, "baseline_model.pt")

if not os.path.exists(BACKUP_PATH):
    torch.save(
        {"model_state_dict": model.state_dict()},
        BACKUP_PATH
    )
    print("Baseline checkpoint backed up:", BACKUP_PATH)
else:
    print("Existing baseline preserved:", BACKUP_PATH)

# Fresh optimizer for continued fine-tuning.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.1
)

print("Device:", device)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Optimizer ready.")
import inspect

print("Model forward signature:")
print(inspect.signature(model.forward))

print("\nBatch function:")
print(inspect.signature(get_batch))

print("\nModel output check:")
x, y = get_batch(train_data, batch_size, block_size, device)
x, y = x.to(device), y.to(device)

model.eval()
with torch.no_grad():
    output = model(x, targets=y)

print("Output type:", type(output))

if isinstance(output, tuple):
    print("Number of returned items:", len(output))
    for i, item in enumerate(output):
        if torch.is_tensor(item):
            print(f"Item {i}: shape={tuple(item.shape)}, value={item.item() if item.numel() == 1 else 'tensor'}")
        else:
            print(f"Item {i}: type={type(item)}")
else:
    print("Output shape:", getattr(output, "shape", "not a tensor"))

model.train()

Existing baseline preserved: /content/drive/MyDrive/TransformerProject/baseline_model.pt
Device: cuda
Trainable parameters: 1713920
Optimizer ready.
Model forward signature:
(input_ids, targets=None, kv_caches=None, position_offset=0)

Batch function:
(data, batch_size, block_size, device)

Model output check:
Output type: <class 'tuple'>
Number of returned items: 7
Item 0: shape=(32, 64, 65), value=tensor
Item 1: shape=(), value=1.6112620830535889
Item 2: shape=(), value=1.5913968086242676
Item 3: shape=(), value=1.9865319728851318
Item 4: type=<class 'int'>
Item 5: type=<class 'list'>
Item 6: type=<class 'list'>


TransformerModel(
  (embedding): Embedding(65, 128)
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (norm1): RMSNorm()
      (attention): GQA(
        (q_proj): Linear(in_features=128, out_features=128, bias=False)
        (k_proj): Linear(in_features=128, out_features=128, bias=False)
        (v_proj): Linear(in_features=128, out_features=128, bias=False)
        (out_proj): Linear(in_features=128, out_features=128, bias=False)
        (rope): RoPE()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): RMSNorm()
      (moe): MoE(
        (router): Linear(in_features=128, out_features=4, bias=False)
        (experts): ModuleList(
          (0-3): 4 x Expert(
            (gate_proj): Linear(in_features=128, out_features=512, bias=False)
            (value_proj): Linear(in_features=128, out_features=512, bias=False)
            (output_proj): Linear(in_features=512, out_features=128, bias=False)
          )
        )
      )
    )
  )
  (final_norm): RMS

In [14]:
import inspect

print(inspect.getsource(type(model).forward))

    def forward(self, input_ids, targets=None, kv_caches=None, position_offset=0):
        x = self.embedding(input_ids)
        total_aux_loss = torch.zeros((), device=x.device)
        total_dropped_tokens = 0
        router_information = []
        new_kv_caches = []

        for i, block in enumerate(self.blocks):
            kv_cache = kv_caches[i] if kv_caches is not None else None
            (
                x,
                new_kv_cache,
                aux_loss,
                router_logits,
                router_probs,
                top_indices,
                dropped_tokens
            ) = block(x, kv_cache=kv_cache, position_offset=position_offset)
            new_kv_caches.append(new_kv_cache)
            total_aux_loss = total_aux_loss + aux_loss
            total_dropped_tokens += dropped_tokens
            router_information.append({
                "logits": router_logits,
                "probs": router_probs,
                "top_indices": top_indices
      

In [16]:

import os
import torch
import math

# Configuration
MAX_STEPS = 1000
EVAL_INTERVAL = 100
EVAL_ITERS = 50
LEARNING_RATE = 1e-4
GRAD_CLIP = 1.0

SAVE_DIR = "/content/drive/MyDrive/TransformerProject"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_PATH = os.path.join(SAVE_DIR, "best_finetuned_model.pt")

# Recreate optimizer for full-model fine-tuning
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.1
)

@torch.no_grad()
def evaluate_model(split, eval_iters=EVAL_ITERS):
    model.eval()
    losses = []
    lm_losses = []
    aux_losses = []

    data = train_data if split == "train" else val_data

    for _ in range(eval_iters):
        x, y = get_batch(data, batch_size, block_size, device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            _, loss, lm_loss, aux_loss, _, _, _ = model(x, targets=y)

        losses.append(loss.item())
        lm_losses.append(lm_loss.item())
        aux_losses.append(aux_loss.item())

    model.train()

    return (
        sum(losses) / len(losses),
        sum(lm_losses) / len(lm_losses),
        sum(aux_losses) / len(aux_losses)
    )

best_val_loss = float("inf")

print("Starting full-model fine-tuning...")
print("Device:", device)
print("Steps:", MAX_STEPS)
print("Learning rate:", LEARNING_RATE)

for step in range(MAX_STEPS):
    model.train()

    x, y = get_batch(train_data, batch_size, block_size, device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        _, loss, lm_loss, aux_loss, dropped, _, _ = model(
            x, targets=y
        )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        train_loss, train_lm, train_aux = evaluate_model("train")
        val_loss, val_lm, val_aux = evaluate_model("val")

        print(
            f"Step {step:4d} | "
            f"Train {train_loss:.4f} | "
            f"Val {val_loss:.4f} | "
            f"Val LM {val_lm:.4f} | "
            f"Val Aux {val_aux:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "step": step,
                    "best_val_loss": best_val_loss,
                    "config": {
                        "learning_rate": LEARNING_RATE,
                        "batch_size": batch_size,
                        "block_size": block_size,
                    },
                },
                BEST_PATH
            )

            print("  Saved improved checkpoint:", BEST_PATH)

print("Fine-tuning complete.")
print(f"Best validation loss: {best_val_loss:.4f}")
print("Best checkpoint:", BEST_PATH)
import torch

BEST_PATH = (
    "/content/drive/MyDrive/TransformerProject/"
    "best_finetuned_model.pt"
)

checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)
model.eval()

print("Loaded best checkpoint")
print("Training step:", checkpoint["step"])
print("Best validation loss:", checkpoint["best_val_loss"])

Starting full-model fine-tuning...
Device: cuda
Steps: 1000
Learning rate: 0.0001
Step    0 | Train 1.6251 | Val 1.8270 | Val LM 1.8072 | Val Aux 1.9885
  Saved improved checkpoint: /content/drive/MyDrive/TransformerProject/best_finetuned_model.pt
Step  100 | Train 1.6181 | Val 1.8081 | Val LM 1.7883 | Val Aux 1.9862
  Saved improved checkpoint: /content/drive/MyDrive/TransformerProject/best_finetuned_model.pt
Step  200 | Train 1.6169 | Val 1.8161 | Val LM 1.7962 | Val Aux 1.9882
Step  300 | Train 1.6071 | Val 1.8166 | Val LM 1.7967 | Val Aux 1.9880
Step  400 | Train 1.6165 | Val 1.8143 | Val LM 1.7944 | Val Aux 1.9872
Step  500 | Train 1.6029 | Val 1.8241 | Val LM 1.8043 | Val Aux 1.9860
Step  600 | Train 1.6067 | Val 1.8157 | Val LM 1.7959 | Val Aux 1.9874
Step  700 | Train 1.5995 | Val 1.8173 | Val LM 1.7974 | Val Aux 1.9852
Step  800 | Train 1.5975 | Val 1.8096 | Val LM 1.7898 | Val Aux 1.9808
Step  900 | Train 1.5985 | Val 1.8029 | Val LM 1.7830 | Val Aux 1.9847
  Saved improved c

In [17]:
prompt_text = "First Citizen:"
prompt_tokens = torch.tensor([encode(prompt_text)], dtype=torch.long, device=device)

MAX_NEW_TOKENS = 120
TEMPERATURE = 0.85
TOP_K = 35
TOP_P=0.90

generated_indices = generate(
    model=model,
    input_ids=prompt_tokens,
    max_new_tokens=MAX_NEW_TOKENS,
    max_seq_len=max_seq_len,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=1.5
)
generated_text = decode(generated_indices[0].tolist())
print("================ GENERATED OUTPUT =================")
print(generated_text)

================ GENERATED OUTPUT =================
First Citizen::
I'll my good,--
Whath waske?

POLIXENES:
Thereful.'ncy, what!'s;
Andubjours.'d,-my prove; but!-Officknow?
Havick,-pay.


In [18]:
BASELINE_PATH = (
    "/content/drive/MyDrive/TransformerProject/"
    "baseline_model.pt"
)

checkpoint = torch.load(
    BASELINE_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model.eval()

print("Original baseline model loaded.")

Original baseline model loaded.


In [19]:
prompt_text = "First Citizen:"
prompt_tokens = torch.tensor([encode(prompt_text)], dtype=torch.long, device=device)

MAX_NEW_TOKENS = 120
TEMPERATURE = 0.85
TOP_K = 35
TOP_P=0.90

generated_indices = generate(
    model=model,
    input_ids=prompt_tokens,
    max_new_tokens=MAX_NEW_TOKENS,
    max_seq_len=max_seq_len,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=1.5
)
generated_text = decode(generated_indices[0].tolist())
print("================ GENERATED OUTPUT =================")
print(generated_text)

================ GENERATED OUTPUT =================
First Citizen:
I'll way, buthough.

PAULINA:
To make,--ford; whence!'s; and,-ppoy?
What!'s.
BROMEO:
Heck, myourngmblives?'dvifff.
Your


In [20]:
print("Dataset vocabulary size:", vocab_size)
print("Model vocabulary size:", model.vocab_size)
print("Block size:", block_size)
print("Training data length:", len(train_data))
print("Validation data length:", len(val_data))

x, y = get_batch(
    train_data, batch_size, block_size, device
)

print("Input shape:", x.shape)
print("Target shape:", y.shape)
print("First input sequence:", x[0, :20].tolist())
print("First target sequence:", y[0, :20].tolist())

print("Shift correct:", torch.equal(x[:, 1:], y[:, :-1]))

Dataset vocabulary size: 65
Model vocabulary size: 65
Block size: 64
Training data length: 1003854
Validation data length: 111540
Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])
First input sequence: [57, 1, 53, 44, 1, 54, 43, 39, 56, 50, 6, 0, 21, 52, 43, 57, 58, 47, 51, 39]
First target sequence: [1, 53, 44, 1, 54, 43, 39, 56, 50, 6, 0, 21, 52, 43, 57, 58, 47, 51, 39, 40]
Shift correct: True


In [21]:

import torch

BEST_PATH = (
    "/content/drive/MyDrive/TransformerProject/"
    "best_finetuned_model.pt"
)
BASELINE_PATH = (
    "/content/drive/MyDrive/TransformerProject/"
    "baseline_model.pt"
)

def evaluate_checkpoint(path, eval_iters=100):
    checkpoint = torch.load(
        path,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )
    model.eval()

    losses = []

    with torch.no_grad():
        for _ in range(eval_iters):
            x, y = get_batch(
                val_data, batch_size, block_size, device
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                _, loss, lm_loss, *_ = model(x, targets=y)

            losses.append(lm_loss.item())

    model.train()
    return sum(losses) / len(losses)

# Use identical batches for both models.
torch.manual_seed(1234)
x_batches = [
    get_batch(val_data, batch_size, block_size, device)
    for _ in range(100)
]

def evaluate_on_fixed_batches(path):
    checkpoint = torch.load(
        path,
        map_location=device,
        weights_only=True
    )
    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )
    model.eval()

    losses = []

    with torch.no_grad():
        for x, y in x_batches:
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                _, _, lm_loss, *_ = model(x, targets=y)

            losses.append(lm_loss.item())

    return sum(losses) / len(losses)

baseline_loss = evaluate_on_fixed_batches(BASELINE_PATH)
finetuned_loss = evaluate_on_fixed_batches(BEST_PATH)

print(f"Baseline validation loss:  {baseline_loss:.4f}")
print(f"Fine-tuned validation loss: {finetuned_loss:.4f}")
print(f"Difference: {baseline_loss - finetuned_loss:.4f}")

Baseline validation loss:  1.8092
Fine-tuned validation loss: 1.7787
Difference: 0.0304


In [22]:

import math
import torch

def evaluate_model_loss(model, eval_iters=300):
    model.eval()
    losses = []

    with torch.no_grad():
        for _ in range(eval_iters):
            x, y = get_batch(
                val_data, batch_size, block_size, device
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                _, _, lm_loss, *_ = model(
                    x, targets=y
                )

            losses.append(lm_loss.item())

    mean_loss = sum(losses) / len(losses)
    perplexity = math.exp(mean_loss)

    model.train()

    print(f"Validation loss: {mean_loss:.4f}")
    print(f"Perplexity:      {perplexity:.4f}")

    return mean_loss, perplexity

# Load the best fine-tuned model before evaluating.
checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

loss, perplexity = evaluate_model_loss(model)

Validation loss: 1.7870
Perplexity:      5.9718


In [23]:

import torch

model.eval()

for length in [64, 128, 256]:
    try:
        x = torch.randint(
            low=0,
            high=model.vocab_size,
            size=(2, length),
            device=device
        )

        with torch.no_grad():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                output = model(x)

        logits = output[0]

        print(
            f"Length {length}: PASS | "
            f"logits shape = {tuple(logits.shape)}"
        )

    except Exception as e:
        print(f"Length {length}: FAILED | {e}")

model.train()

Length 64: PASS | logits shape = (2, 64, 65)
Length 128: PASS | logits shape = (2, 128, 65)
Length 256: PASS | logits shape = (2, 256, 65)


TransformerModel(
  (embedding): Embedding(65, 128)
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (norm1): RMSNorm()
      (attention): GQA(
        (q_proj): Linear(in_features=128, out_features=128, bias=False)
        (k_proj): Linear(in_features=128, out_features=128, bias=False)
        (v_proj): Linear(in_features=128, out_features=128, bias=False)
        (out_proj): Linear(in_features=128, out_features=128, bias=False)
        (rope): RoPE()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): RMSNorm()
      (moe): MoE(
        (router): Linear(in_features=128, out_features=4, bias=False)
        (experts): ModuleList(
          (0-3): 4 x Expert(
            (gate_proj): Linear(in_features=128, out_features=512, bias=False)
            (value_proj): Linear(in_features=128, out_features=512, bias=False)
            (output_proj): Linear(in_features=512, out_features=128, bias=False)
          )
        )
      )
    )
  )
  (final_norm): RMS

In [24]:

import torch
import math
import os

# Keep the current architecture and sequence length.
BLOCK_SIZE = 64
MAX_STEPS = 2000
EVAL_INTERVAL = 200
EVAL_ITERS = 50

MAX_LR = 5e-5
MIN_LR = 1e-5
WARMUP_STEPS = 100
GRAD_CLIP = 1.0

BEST_PATH = (
    "/content/drive/MyDrive/TransformerProject/"
    "best_finetuned_model.pt"
)

CONTINUED_PATH = (
    "/content/drive/MyDrive/TransformerProject/"
    "continued_training.pt"
)

assert os.path.exists(BEST_PATH), (
    f"Checkpoint not found: {BEST_PATH}"
)

checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model.to(device)
model.train()

# Fresh optimizer for this training phase.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    weight_decay=0.1
)

print("Checkpoint loaded successfully.")
print("Device:", device)
print("Training steps:", MAX_STEPS)
print("Block size:", BLOCK_SIZE)
print("Learning rate:", MAX_LR)
print("Model parameters:", sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
))

Checkpoint loaded successfully.
Device: cuda
Training steps: 2000
Block size: 64
Learning rate: 5e-05
Model parameters: 1713920


In [25]:

best_val_loss = float("inf")
history = []

def get_lr(step):
    if step < WARMUP_STEPS:
        return MAX_LR * (step + 1) / WARMUP_STEPS

    progress = min(
        1.0,
        (step - WARMUP_STEPS)
        / (MAX_STEPS - WARMUP_STEPS)
    )

    return MIN_LR + 0.5 * (MAX_LR - MIN_LR) * (
        1 + math.cos(math.pi * progress)
    )

@torch.no_grad()
def estimate_losses():
    model.eval()
    results = {}

    for split, data in [
        ("train", train_data),
        ("val", val_data)
    ]:
        losses = []

        for _ in range(EVAL_ITERS):
            x, y = get_batch(
                data, batch_size, BLOCK_SIZE, device
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                _, _, lm_loss, *_ = model(
                    x, targets=y
                )

            losses.append(lm_loss.item())

        results[split] = sum(losses) / len(losses)

    model.train()
    return results

print("Starting continued Transformer training...")

for step in range(MAX_STEPS):
    lr = get_lr(step)

    for group in optimizer.param_groups:
        group["lr"] = lr

    x, y = get_batch(
        train_data, batch_size, BLOCK_SIZE, device
    )

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        _, loss, lm_loss, aux_loss, *_ = model(
            x, targets=y
        )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(), GRAD_CLIP
    )

    optimizer.step()

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        metrics = estimate_losses()

        train_loss = metrics["train"]
        val_loss = metrics["val"]

        history.append({
            "step": step,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": lr
        })

        print(
            f"Step {step:4d} | "
            f"Train {train_loss:.4f} | "
            f"Val {val_loss:.4f} | "
            f"LR {lr:.2e}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "step": step,
                "best_val_loss": best_val_loss,
                "block_size": BLOCK_SIZE
            }, CONTINUED_PATH)

            print("  Saved improved checkpoint.")

print("\nTraining complete.")
print(f"Best validation loss: {best_val_loss:.4f}")
print("Best checkpoint:", CONTINUED_PATH)

Starting continued Transformer training...
Step    0 | Train 1.5897 | Val 1.7930 | LR 5.00e-07
  Saved improved checkpoint.
Step  200 | Train 1.5715 | Val 1.7842 | LR 4.97e-05
  Saved improved checkpoint.
Step  400 | Train 1.5593 | Val 1.7787 | LR 4.76e-05
  Saved improved checkpoint.
Step  600 | Train 1.5716 | Val 1.7980 | LR 4.35e-05
Step  800 | Train 1.5601 | Val 1.7900 | LR 3.80e-05
Step 1000 | Train 1.5603 | Val 1.7683 | LR 3.17e-05
  Saved improved checkpoint.
Step 1200 | Train 1.5690 | Val 1.7794 | LR 2.51e-05
Step 1400 | Train 1.5567 | Val 1.7792 | LR 1.91e-05
Step 1600 | Train 1.5543 | Val 1.7700 | LR 1.42e-05
Step 1800 | Train 1.5529 | Val 1.7724 | LR 1.11e-05
Step 1999 | Train 1.5575 | Val 1.7680 | LR 1.00e-05
  Saved improved checkpoint.

Training complete.
Best validation loss: 1.7680
Best checkpoint: /content/drive/MyDrive/TransformerProject/continued_training.pt


In [26]:

BEST_CONTINUED_PATH = (
    "/content/drive/MyDrive/TransformerProject/"
    "continued_training.pt"
)

checkpoint = torch.load(
    BEST_CONTINUED_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model.eval()

print("Best continued-training checkpoint loaded.")
print("Training step:", checkpoint["step"])
print("Best validation loss:", checkpoint["best_val_loss"])
prompt_text = "First Citizen:"
prompt_tokens = torch.tensor([encode(prompt_text)], dtype=torch.long, device=device)

MAX_NEW_TOKENS = 120
TEMPERATURE = 0.85
TOP_K = 35
TOP_P=0.90

generated_indices = generate(
    model=model,
    input_ids=prompt_tokens,
    max_new_tokens=MAX_NEW_TOKENS,
    max_seq_len=max_seq_len,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=1.5
)
generated_text = decode(generated_indices[0].tolist())
print("================ GENERATED OUTPUT =================")
print(generated_text)

Best continued-training checkpoint loaded.
Training step: 1999
Best validation loss: 1.768037874698639
================ GENERATED OUTPUT =================
First Citizen::
I'll my good,--
Whath before, what's.

POMPEY:
Tulk!'s; Spray?
Anduckengemby,--forchive.'d; what!'Glf?
Bucknowives; RI


In [31]:


import torch
import os

# Preserve the current best model.
PREVIOUS_BEST = (
    "/content/drive/MyDrive/TransformerProject/"
    "continued_training.pt"
)

NEXT_BEST = (
    "/content/drive/MyDrive/TransformerProject/"
    "context128_best.pt"
)

checkpoint = torch.load(
    PREVIOUS_BEST,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model.to(device)
model.train()

# Increase training context.
BLOCK_SIZE = 256

# Start with a fresh optimizer for this phase.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5,
    weight_decay=0.1
)

print("Starting checkpoint loaded.")
print("Training context:", BLOCK_SIZE)
print("Learning rate:", 3e-5)
print("Device:", device)
import math
import os
import torch

MAX_STEPS = 2000
EVAL_INTERVAL = 200
EVAL_ITERS = 50
BEST_VAL = float("inf")

# Define the missing NEXT_BEST path variable
NEXT_BEST = "/content/drive/MyDrive/TransformerProject/best_model_context_128.pt"
os.makedirs(os.path.dirname(NEXT_BEST), exist_ok=True)

def evaluate_lm_loss(data):
    model.eval()
    losses = []

    with torch.no_grad():
        for _ in range(EVAL_ITERS):
            x, y = get_batch(
                data, batch_size, BLOCK_SIZE, device
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                _, _, lm_loss, *_ = model(
                    x, targets=y
                )

            losses.append(lm_loss.item())

    model.train()
    return sum(losses) / len(losses)

print("Training with context length 128...")

for step in range(MAX_STEPS):
    # Cosine learning-rate decay.
    progress = step / MAX_STEPS
    lr = 3e-5 * (
        0.2 + 0.8 * 0.5 * (1 + math.cos(math.pi * progress))
    )

    for group in optimizer.param_groups:
        group["lr"] = lr

    x, y = get_batch(
        train_data, batch_size, BLOCK_SIZE, device
    )

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        _, loss, lm_loss, *_ = model(x, targets=y)

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(), 1.0
    )

    optimizer.step()

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        train_loss = evaluate_lm_loss(train_data)
        val_loss = evaluate_lm_loss(val_data)

        print(
            f"Step {step:4d} | "
            f"Train {train_loss:.4f} | "
            f"Val {val_loss:.4f} | "
            f"LR {lr:.2e}"
        )

        if val_loss < BEST_VAL:
            BEST_VAL = val_loss

            torch.save({
                "model_state_dict": model.state_dict(),
                "step": step,
                "best_val_loss": BEST_VAL,
                "block_size": BLOCK_SIZE
            }, NEXT_BEST)

            print("  Saved improved checkpoint.")

print("Context-128 training complete.")
print("Best validation loss:", BEST_VAL)
print("Checkpoint:", NEXT_BEST)
prompt_text = "First Citizen:"
prompt_tokens = torch.tensor([encode(prompt_text)], dtype=torch.long, device=device)

MAX_NEW_TOKENS = 120
TEMPERATURE = 0.85
TOP_K = 35
TOP_P=0.90

generated_indices = generate(
    model=model,
    input_ids=prompt_tokens,
    max_new_tokens=MAX_NEW_TOKENS,
    max_seq_len=max_seq_len,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=1.5
)
generated_text = decode(generated_indices[0].tolist())
print("================ GENERATED OUTPUT =================")
print(generated_text)

Starting checkpoint loaded.
Training context: 256
Learning rate: 3e-05
Device: cuda
Training with context length 128...
Step    0 | Train 2.1683 | Val 2.3127 | LR 3.00e-05
  Saved improved checkpoint.
Step  200 | Train 1.6310 | Val 1.8211 | LR 2.94e-05
  Saved improved checkpoint.
Step  400 | Train 1.5718 | Val 1.7785 | LR 2.77e-05
  Saved improved checkpoint.
Step  600 | Train 1.5600 | Val 1.7650 | LR 2.51e-05
  Saved improved checkpoint.
Step  800 | Train 1.5385 | Val 1.7621 | LR 2.17e-05
  Saved improved checkpoint.
Step 1000 | Train 1.5345 | Val 1.7672 | LR 1.80e-05
Step 1200 | Train 1.5239 | Val 1.7684 | LR 1.43e-05
Step 1400 | Train 1.5270 | Val 1.7452 | LR 1.09e-05
  Saved improved checkpoint.
Step 1600 | Train 1.5249 | Val 1.7664 | LR 8.29e-06
Step 1800 | Train 1.5241 | Val 1.7595 | LR 6.59e-06
Step 1999 | Train 1.5195 | Val 1.7681 | LR 6.00e-06
Context-128 training complete.
Best validation loss: 1.7452274990081786
Checkpoint: /content/drive/MyDrive/TransformerProject/best_mod

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
%pip install -q -U transformers peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [5]:
import torch

# Original weights in 32-bit floating point
weights = torch.tensor(
    [0.12, -0.45, 0.78, 1.23, -1.10, 0.34],
    dtype=torch.float32
)

# Simulate reduced precision using float16
weights_fp16 = weights.to(torch.float16)

print("Original weights:", weights)
print("FP16 weights:    ", weights_fp16)

print("FP32 storage per value:", weights.element_size(), "bytes")
print("FP16 storage per value:", weights_fp16.element_size(), "bytes")
weights_restored = weights_fp16.float()

error = (weights - weights_restored).abs()

print("Restored weights:", weights_restored)
print("Absolute errors:", error)
print("Maximum absolute error:", error.max().item())
import torch

weights = torch.tensor(
    [0.12, -0.45, 0.78, 1.23, -1.10, 0.34],
    dtype=torch.float32
)

# Choose a symmetric range based on the largest magnitude
scale = weights.abs().max() / 7

# Quantize to signed integer levels from -7 to +7
quantized = torch.round(weights / scale).clamp(-7, 7)

# Reconstruct approximate floating-point values
reconstructed = quantized * scale

print("Original:      ", weights)
print("Quantized:     ", quantized)
print("Reconstructed: ", reconstructed)
print("Absolute error:", (weights - reconstructed).abs())

Original weights: tensor([ 0.1200, -0.4500,  0.7800,  1.2300, -1.1000,  0.3400])
FP16 weights:     tensor([ 0.1200, -0.4500,  0.7798,  1.2305, -1.0996,  0.3401],
       dtype=torch.float16)
FP32 storage per value: 4 bytes
FP16 storage per value: 2 bytes
Restored weights: tensor([ 0.1200, -0.4500,  0.7798,  1.2305, -1.0996,  0.3401])
Absolute errors: tensor([4.8801e-06, 4.8816e-05, 2.1482e-04, 4.6873e-04, 3.9065e-04, 8.7887e-05])
Maximum absolute error: 0.0004687309265136719
Original:       tensor([ 0.1200, -0.4500,  0.7800,  1.2300, -1.1000,  0.3400])
Quantized:      tensor([ 1., -3.,  4.,  7., -6.,  2.])
Reconstructed:  tensor([ 0.1757, -0.5271,  0.7029,  1.2300, -1.0543,  0.3514])
Absolute error: tensor([0.0557, 0.0771, 0.0771, 0.0000, 0.0457, 0.0114])


In [6]:
import torch
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(quant_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
)

print("Model loaded successfully")
print("Parameters:", f"{model.num_parameters():,}")
print("Embedding device:", model.get_input_embeddings().weight.device)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded successfully
Parameters: 494,032,768
Embedding device: cuda:0


In [8]:
import torch
import bitsandbytes as bnb

print("Model class:", type(model).__name__)
print("4-bit quantization enabled:", model.is_loaded_in_4bit)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Currently trainable: {trainable_params:,}")

# Find a quantized linear layer
for name, module in model.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        print("\nExample quantized layer:", name)
        print("Layer class:", type(module).__name__)
        print("Quantization state:", module.weight.quant_state)
        break

Model class: Qwen2ForCausalLM
4-bit quantization enabled: True
Total parameters: 315,119,488
Currently trainable: 136,206,208

Example quantized layer: model.layers.0.self_attn.q_proj
Layer class: Linear4bit
Quantization state: <bitsandbytes.functional.QuantState object at 0x7ba6e12be120>


In [9]:
prompt = "Explain quantum computing in one simple sentence."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize and generate
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

print("Generating response...")
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True
)

# Trim the input tokens from the output response
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("\n--- Output ---")
print(response)

Generating response...

--- Output ---
Quantum computing is a new type of computing device that uses quantum bits, also called qubits. These qubits can store multiple information simultaneously and can perform operations in parallel, which is not possible with classical computers. Quantum computing has the potential to revolutionize various industries, including cryptography, drug discovery, and optimization problems.
 authDomain
You are a helpful assistant.ufig
Explain quantum computing in one simple sentence.ufigassistant
Quantum computing is a new and innovative way of computing, which uses


In [10]:
print(len(model.model.layers))

24


In [11]:
from transformers import AutoConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Load the configuration object
config = AutoConfig.from_pretrained(MODEL_ID)

# 2. Print the entire configuration directly (displays all hyperparameters cleanly)
print("--- Full Model Configuration ---")
print(config)

# 3. Alternatively, convert it to a standard Python dictionary to inspect or loop through keys
print("\n--- Sorted Hyperparameters ---")
config_dict = config.to_dict()
for key, value in sorted(config_dict.items()):
    print(f"{key}: {value}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

--- Full Model Configuration ---
Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  

In [12]:
trainable = 0
total = 0

for name, param in model.named_parameters():
    total += param.numel()

    if param.requires_grad:
        trainable += param.numel()
        print(
            f"{name:70s} "
            f"{param.numel():>12,}"
        )

print("\nTrainable parameters:", f"{trainable:,}")
print("Total parameters:", f"{total:,}")
print("Trainable percentage:", f"{100 * trainable / total:.2f}%")

model.embed_tokens.weight                                               136,134,656
model.layers.0.self_attn.q_proj.bias                                            896
model.layers.0.self_attn.k_proj.bias                                            128
model.layers.0.self_attn.v_proj.bias                                            128
model.layers.0.input_layernorm.weight                                           896
model.layers.0.post_attention_layernorm.weight                                  896
model.layers.1.self_attn.q_proj.bias                                            896
model.layers.1.self_attn.k_proj.bias                                            128
model.layers.1.self_attn.v_proj.bias                                            128
model.layers.1.input_layernorm.weight                                           896
model.layers.1.post_attention_layernorm.weight                                  896
model.layers.2.self_attn.q_proj.bias                                        